In [39]:
import json
import sqlite3
from typing import List, Dict, Any
from openai import OpenAI
import os
import requests as r

# Fetch data from the OData API
url = 'https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=ShipCountry%20eq%20%27Germany%27%20and%20Freight%20gt%2020&$format=json'
response = r.get(url)
res1 = response.json()
json_data = res1['value']

class JSONalyze:
    def __init__(self):
        self.conn = sqlite3.connect(':memory:')
        self.cursor = self.conn.cursor()
        self.table_name = 'json_data'
        self.columns = []

        # Set up OpenAI client
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        if not self.client:
            raise ValueError("OpenAI API key not set in environment variables.")

    def load_json(self, json_data: List[Dict[str, Any]]):
        if not json_data:
            raise ValueError("JSON data is empty")

        self.columns = list(json_data[0].keys())

        # Create table
        columns = ', '.join([f'"{k}" TEXT' for k in self.columns])
        self.cursor.execute(f'CREATE TABLE {self.table_name} ({columns})')

        # Insert data
        placeholders = ', '.join(['?' for _ in json_data[0]])
        for item in json_data:
            values = [json.dumps(v) if isinstance(v, (dict, list)) else v for v in item.values()]
            self.cursor.execute(f'INSERT INTO {self.table_name} VALUES ({placeholders})', values)

        self.conn.commit()

    def natural_language_to_sql(self, query: str) -> str:
        prompt = f"""
        Convert the following natural language query to SQL. 
        The table name is '{self.table_name}' and it has the following columns: {', '.join(self.columns)}.
        Provide only the SQL query without any additional formatting or explanation.
        
        Natural language query: {query}
        
        SQL query:
        """

        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that converts natural language queries to SQL. Provide only the SQL query without any additional formatting or explanation."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            n=1,
            stop=None,
            temperature=0.1,
        )

        if response.choices:
            sql_query = response.choices[0].message.content.strip()
            sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
            return sql_query
        else:
            raise ValueError("No SQL query returned from OpenAI API.")

    def execute_sql(self, sql_query: str) -> str:
        print(f"Executing SQL query: {sql_query}")  # For debugging

        try:
            self.cursor.execute(sql_query)
            columns = [description[0] for description in self.cursor.description]
            results = [dict(zip(columns, row)) for row in self.cursor.fetchall()]

            # Format the output
            if results:
                output = '\n'.join(', '.join(f"{key}: {value}" for key, value in row.items()) for row in results)
                return f"Results:\n{output}"
            else:
                return "No results found."
        except sqlite3.Error as e:
            return f"An error occurred: {e}"

    def print_table(self):
        try:
            self.cursor.execute(f'SELECT * FROM {self.table_name}')
            columns = [description[0] for description in self.cursor.description]
            rows = self.cursor.fetchall()

            # Print column names
            print(f"Table: {self.table_name}")
            print(f"{' | '.join(columns)}")
            print('-' * (len(columns) * 20))

            # Print rows
            for row in rows:
                print(' | '.join(str(cell) for cell in row))
        except sqlite3.Error as e:
            print(f"An error occurred: {e}")

    def close(self):
        self.conn.close()

# Example usage
if __name__ == "__main__":
    jsonalyze = JSONalyze()
    jsonalyze.load_json(json_data)

    # Print the entire table
    # jsonalyze.print_table()

    # Execute a direct SQL query and print the result
    sql_query = "SELECT * FROM json_data WHERE ShipCountry = 'Germany' AND Freight > 600;"
    print(jsonalyze.execute_sql(sql_query))

    # Alternatively, execute a query using natural language
    natural_query = "How many orders sent to Germany have freight cost greater than 600 $"
    sql_query_from_nl = jsonalyze.natural_language_to_sql(natural_query)
    print(jsonalyze.execute_sql(sql_query_from_nl))

    print("\nFreight data types:")
    result = jsonalyze.execute_sql("SELECT DISTINCT typeof(Freight) FROM json_data")
    print(result)

    jsonalyze.close()


Executing SQL query: SELECT * FROM json_data WHERE ShipCountry = 'Germany' AND Freight > 600;
Results:
OrderID: 10273, CustomerID: QUICK, EmployeeID: 3, OrderDate: 1996-08-05T00:00:00Z, RequiredDate: 1996-09-02T00:00:00Z, ShippedDate: 1996-08-12T00:00:00Z, ShipVia: 3, Freight: 76.07, ShipName: QUICK-Stop, ShipAddress: Taucherstraße 10, ShipCity: Cunewalde, ShipRegion: None, ShipPostalCode: 01307, ShipCountry: Germany
OrderID: 10284, CustomerID: LEHMS, EmployeeID: 4, OrderDate: 1996-08-19T00:00:00Z, RequiredDate: 1996-09-16T00:00:00Z, ShippedDate: 1996-08-27T00:00:00Z, ShipVia: 1, Freight: 76.56, ShipName: Lehmanns Marktstand, ShipAddress: Magazinweg 7, ShipCity: Frankfurt a.M., ShipRegion: None, ShipPostalCode: 60528, ShipCountry: Germany
OrderID: 10285, CustomerID: QUICK, EmployeeID: 1, OrderDate: 1996-08-20T00:00:00Z, RequiredDate: 1996-09-17T00:00:00Z, ShippedDate: 1996-08-26T00:00:00Z, ShipVia: 2, Freight: 76.83, ShipName: QUICK-Stop, ShipAddress: Taucherstraße 10, ShipCity: Cunewa

In [6]:
import json
import sqlite3
from typing import List, Dict, Any
from openai import OpenAI
import os
import requests as r

# Fetch data from the OData API
url = 'https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=ShipCountry%20eq%20%27Germany%27%20and%20Freight%20gt%2020&$format=json'
response = r.get(url)
res1 = response.json()
json_data = res1['value']

class JSONalyze:
    def __init__(self):
        self.conn = sqlite3.connect(':memory:')
        self.cursor = self.conn.cursor()
        self.table_name = 'json_data'
        self.columns = []

        # Set up OpenAI client
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        if not self.client:
            raise ValueError("OpenAI API key not set in environment variables.")

    def load_json(self, json_data: List[Dict[str, Any]]):
        if not json_data:
            raise ValueError("JSON data is empty")

        self.columns = list(json_data[0].keys())

        # Create table with TEXT type for all columns
        columns = ', '.join([f'"{k}" TEXT' for k in self.columns])
        self.cursor.execute(f'CREATE TABLE {self.table_name} ({columns})')

        # Insert data
        placeholders = ', '.join(['?' for _ in self.columns])
        for item in json_data:
            values = [item.get(k) for k in self.columns]
            self.cursor.execute(f'INSERT INTO {self.table_name} VALUES ({placeholders})', values)

        self.conn.commit()

    def inspect_column_types(self):
        self.cursor.execute(f'PRAGMA table_info({self.table_name})')
        return {row[1]: row[2] for row in self.cursor.fetchall()}  # {column_name: data_type}

    def natural_language_to_sql(self, query: str) -> str:
        prompt = f"""
        Convert the following natural language query to SQL. 
        The table name is '{self.table_name}' and it has the following columns: {', '.join(self.columns)}.
        Provide only the SQL query without any additional formatting or explanation.
        
        Natural language query: {query}
        
        SQL query:
        """

        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that converts natural language queries to SQL. Provide only the SQL query without any additional formatting or explanation."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            n=1,
            stop=None,
            temperature=0.1,
        )

        if response.choices:
            sql_query = response.choices[0].message.content.strip()
            sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
            return sql_query
        else:
            raise ValueError("No SQL query returned from OpenAI API.")

    def adjust_query_for_types(self, sql_query: str) -> str:
        column_types = self.inspect_column_types()

        for column, data_type in column_types.items():
            if data_type == 'TEXT' and column in sql_query:
                if '>' in sql_query or '<' in sql_query:
                    sql_query = sql_query.replace(f"{column} >", f"CAST({column} AS REAL) >")
                    sql_query = sql_query.replace(f"{column} <", f"CAST({column} AS REAL) <")

        return sql_query

    def execute_sql(self, sql_query: str) -> str:
        sql_query_adjusted = self.adjust_query_for_types(sql_query)
        print(f"Executing SQL query: {sql_query_adjusted}")  # For debugging

        try:
            self.cursor.execute(sql_query_adjusted)
            columns = [description[0] for description in self.cursor.description]
            results = [dict(zip(columns, row)) for row in self.cursor.fetchall()]

            # Format the output
            if results:
                output = '\n'.join(', '.join(f"{key}: {value}" for key, value in row.items()) for row in results)
                return f"Results:\n{output}"
            else:
                return "No results found."
        except sqlite3.Error as e:
            return f"An error occurred: {e}"

    def print_table(self):
        try:
            self.cursor.execute(f'SELECT * FROM {self.table_name}')
            columns = [description[0] for description in self.cursor.description]
            rows = self.cursor.fetchall()

            # Print column names
            print(f"Table: {self.table_name}")
            print(f"{' | '.join(columns)}")
            print('-' * (len(columns) * 20))

            # Print rows
            for row in rows:
                print(' | '.join(str(cell) for cell in row))
        except sqlite3.Error as e:
            print(f"An error occurred: {e}")

    def close(self):
        self.conn.close()

# Example usage
if __name__ == "__main__":
    jsonalyze = JSONalyze()
    jsonalyze.load_json(json_data)

    # Print the entire table
    # jsonalyze.print_table()

    # # Execute a direct SQL query and print the result
    # sql_query = "SELECT * FROM json_data WHERE ShipCountry = 'Germany' AND Freight > 600;"
    # print(jsonalyze.execute_sql(sql_query))

    # Alternatively, execute a query using natural language
    natural_query = "How many orders sent to Germany have freight cost greater than 600 $"
    sql_query_from_nl = jsonalyze.natural_language_to_sql(natural_query)
    print(jsonalyze.execute_sql(sql_query_from_nl))

    jsonalyze.close()


Executing SQL query: SELECT COUNT(*)
FROM json_data
WHERE ShipCountry = 'Germany' AND CAST(Freight AS REAL) > 600;
Results:
COUNT(*): 2


Executing SQL query: SELECT COUNT(*) 
FROM json_data 
WHERE EmployeeID = 3 
AND DATE(ShippedDate) <= RequiredDate;
Results:
COUNT(*): 122


In [1]:
import json
import sqlite3
from typing import List, Dict, Any
from openai import OpenAI
import os
import requests as r

# Fetch data from the OData API
url = 'https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=ShipVia eq 1 and RequiredDate le ShippedDate&$format=json'
response = r.get(url)
res1 = response.json()
json_data = res1['value']

class JSONalyze:
    def __init__(self):
        self.conn = sqlite3.connect(':memory:')
        self.cursor = self.conn.cursor()
        self.table_name = 'json_data'
        self.columns = []

        # Set up OpenAI client
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        if not self.client:
            raise ValueError("OpenAI API key not set in environment variables.")

    def load_json(self, json_data: List[Dict[str, Any]]):
        if not json_data:
            raise ValueError("JSON data is empty")

        self.columns = list(json_data[0].keys())

        # Create table with TEXT type for all columns
        columns = ', '.join([f'"{k}" TEXT' for k in self.columns])
        self.cursor.execute(f'CREATE TABLE {self.table_name} ({columns})')

        # Insert data
        placeholders = ', '.join(['?' for _ in self.columns])
        for item in json_data:
            values = [item.get(k) for k in self.columns]
            self.cursor.execute(f'INSERT INTO {self.table_name} VALUES ({placeholders})', values)

        self.conn.commit()

    def inspect_column_types(self):
        self.cursor.execute(f'PRAGMA table_info({self.table_name})')
        return {row[1]: row[2] for row in self.cursor.fetchall()}  # {column_name: data_type}

    def query_llm_for_types(self, query: str) -> str:
        prompt = f"""
        Adjust the following SQL query to handle the correct data types for columns in the table '{self.table_name}' with the following columns: {', '.join(self.columns)}.
        The data types can be either TEXT, REAL (for numbers), or DATE (for dates).
        Ensure the query handles comparisons and operations appropriately for each data type.
        
        SQL query: {query}
        
        Adjusted SQL query:
        """

        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that adjusts SQL queries to handle correct data types for columns in a table."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            n=1,
            stop=None,
            temperature=0.1,
        )

        if response.choices:
            adjusted_sql_query = response.choices[0].message.content.strip()
            adjusted_sql_query = adjusted_sql_query.replace('```sql', '').replace('```', '').strip()
            return adjusted_sql_query
        else:
            raise ValueError("No SQL query returned from OpenAI API.")

    def natural_language_to_sql(self, query: str) -> str:
        prompt = f"""
        Convert the following natural language query to SQL. 
        The table name is '{self.table_name}' and it has the following columns: {', '.join(self.columns)}.
        Provide only the SQL query without any additional formatting or explanation.
        
        Natural language query: {query}
        
        SQL query:
        """

        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that converts natural language queries to SQL. Provide only the SQL query without any additional formatting or explanation."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            n=1,
            stop=None,
            temperature=0.1,
        )

        if response.choices:
            sql_query = response.choices[0].message.content.strip()
            sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
            return sql_query
        else:
            raise ValueError("No SQL query returned from OpenAI API.")

    def query(self, natural_query: str) -> str:
        sql_query = self.natural_language_to_sql(natural_query)
        sql_query_adjusted = self.query_llm_for_types(sql_query)
        print(f"Executing SQL query: {sql_query_adjusted}")  # For debugging

        try:
            self.cursor.execute(sql_query_adjusted)
            columns = [description[0] for description in self.cursor.description]
            results = [dict(zip(columns, row)) for row in self.cursor.fetchall()]

            # Format the output
            if results:
                output = '\n'.join(', '.join(f"{key}: {value}" for key, value in row.items()) for row in results)
                return f"Results:\n{output}"
            else:
                return "No results found."
        except sqlite3.Error as e:
            return f"An error occurred: {e}"

    def close(self):
        self.conn.close()

# Example usage
if __name__ == "__main__":
    jsonalyze = JSONalyze()
    jsonalyze.load_json(json_data)

    # Execute a query using natural language
    natural_query = "Which orders were shipped using ShipVia ID 1 and have been delayed?"
    print(jsonalyze.query(natural_query))

    jsonalyze.close()


Executing SQL query: SELECT * 
FROM json_data 
WHERE ShipVia = '1' 
AND DATE(ShippedDate) > DATE(RequiredDate);
Results:
OrderID: 10280, CustomerID: BERGS, EmployeeID: 2, OrderDate: 1996-08-14T00:00:00Z, RequiredDate: 1996-09-11T00:00:00Z, ShippedDate: 1996-09-12T00:00:00Z, ShipVia: 1, Freight: 8.98, ShipName: Berglunds snabbköp, ShipAddress: Berguvsvägen  8, ShipCity: Luleå, ShipRegion: None, ShipPostalCode: S-958 22, ShipCountry: Sweden
OrderID: 10309, CustomerID: HUNGO, EmployeeID: 3, OrderDate: 1996-09-19T00:00:00Z, RequiredDate: 1996-10-17T00:00:00Z, ShippedDate: 1996-10-23T00:00:00Z, ShipVia: 1, Freight: 47.3, ShipName: Hungry Owl All-Night Grocers, ShipAddress: 8 Johnstown Road, ShipCity: Cork, ShipRegion: Co. Cork, ShipPostalCode: None, ShipCountry: Ireland
OrderID: 10515, CustomerID: QUICK, EmployeeID: 2, OrderDate: 1997-04-23T00:00:00Z, RequiredDate: 1997-05-07T00:00:00Z, ShippedDate: 1997-05-23T00:00:00Z, ShipVia: 1, Freight: 204.47, ShipName: QUICK-Stop, ShipAddress: Tauche